In [2]:
%load_ext autoreload
%autoreload 2

### Install SDG
```bash 
git clone https://github.com/Red-Hat-AI-Innovation-Team/sdg_hub.git
cd sdg_hub
pip install .[examples]
copy the .env.example to .env and set the model endpoint and generation/mixing parameters
```
**⚠️ If you haven't already, run the document pre-processing notebook to create the seed data.**

In [3]:
# Third Party
from datasets import load_dataset
from dotenv import load_dotenv

# First Party
from sdg_hub import Flow, FlowRegistry
import os

# Load environment variables from .env file
load_dotenv()

/Users/mathale/redhat-projects/sdg_hub/test_nb/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


True

In [4]:
# Required to run the flow with async mode
import nest_asyncio

nest_asyncio.apply()  

In [5]:
def create_seed_data(run_on_validation=None, seed_data_path=None):
    """
    Create seed data from QuALITY Benchmark dataset.
    
    Args:
        run_on_validation (bool, optional): If True, use validation subset. If None, reads from env.
        seed_data_path (str, optional): Path to save seed data. If None, reads from env.
    
    Returns:
        datasets.Dataset: The processed corpus
    """
    # Use environment variables as defaults if not provided
    if run_on_validation is None:
        run_on_validation = os.getenv('RUN_ON_VALIDATION_SET', 'true').lower() == 'true'
    if seed_data_path is None:
        seed_data_path = os.getenv('SEED_DATA_PATH', 'seed_data_val.jsonl')
    
    # Load QuALITY Benchmark dataset
    print("Loading QuALITY Benchmark dataset...")
    quality_corpus = load_dataset("zitongyang/entigraph-quality-corpus", split='train').remove_columns(['entity', 'entigraph']).rename_columns({'raw': 'document', 'uid': 'document_outline'})
    
    # Define seed examples for knowledge tuning
    seed_examples = {
        "icl_document": (
          "The coastal town of Willow Creek, once renowned for its pristine beaches, now struggles with rampant pollution. Plastic debris and oil spills have devastated marine life, prompting a decline in tourism and fishing industries. Residents have organized weekly clean-up initiatives, but the scale of the problem overwhelms their efforts.",
          "Technologists at the local university have developed an AI-powered buoy system to combat this. The buoys, equipped with solar panels and filtration technology, can identify and absorb oil spills while collecting microplastics. Data from the buoys is shared publicly, raising awareness and pressuring corporations to adopt sustainable practices. Though costly, the project has sparked hope for revitalizing the ecosystem and economy."
        ),
        "icl_query_1": "How does the technological solution address the economic *and* environmental challenges highlighted in the document?",
        "icl_query_2": "What implicit values or priorities do the community's actions (clean-up initiatives) and the technologists' project reflect, and how do these align or contrast?",
        "icl_query_3": "Imagine the buoy project succeeds. What unintended consequences might arise from its impact, considering document's themes?",
        "domain": "articles/essays"
    }
    
    # Add seed examples to the corpus
    quality_corpus = quality_corpus.map(lambda x: seed_examples)
    
    if run_on_validation:
        # Validation set - use predefined document IDs for consistent evaluation
        DOC_UIDS = [
            ' Defining Decay Down by David Plotz',
            ' Fight Clubbed by David Plotz',
            ' I, Antichrist? by Jeffrey Goldberg',
            " It's Time To Keelhaul U-Haul! by Jeffrey Goldberg",
            " My Father's Estate by Ben Stein",
            '"Phone Me in Central Park" by McConnell, James V.',
            'A Coffin for Jacob by Ludwig, Edward W.',
            'A Fall of Glass by Lee, Stanley R.',
            'A Filbert Is a Nut by Raphael, Rick',
            'A Gift from Earth by Banister, Manly',
            'A Gleeb for Earth by Schafhauser, Charles',
            'A Good Year for the Roses? by David Edelstein',
            'A Pail of Air by Leiber, Fritz',
            'A Planet Named Joe by Hunter, Evan',
            "AI: what's the worst that could happen? by Harry Armstrong",
            'Accidental Death by Baily, Peter',
            'All Day September by Kuykendall, Roger',
            'Ambition by Bade, William L.',
            'And Then the Town Took Off by Wilson, Richard',
            'Atom Mystery [Young Atom Detective] by Coombs, Charles Ira',
            'Beach Scene by King, Marshall',
            'Big Ancestor by Wallace, F. L. (Floyd L.)',
            'Birds of a Feather by Silverberg, Robert',
            'Bodyguard by Gold, H. L. (Horace Leonard)'
        ]
        
        # Filter corpus to validation set
        quality_corpus = quality_corpus.filter(lambda x: x['document_outline'] in DOC_UIDS)
        print(f"Running on validation set with {len(quality_corpus)} documents")
    else:
        # Use full dataset for training
        print(f"Running on full dataset with {len(quality_corpus)} documents")
    
    # Save the seed data
    quality_corpus.to_json(seed_data_path, orient='records', lines=True)
    print(f"Saved seed data to: {seed_data_path}")
    
    return quality_corpus

In [6]:
# Create seed data using the function
quality_corpus = create_seed_data().select([1])

Loading QuALITY Benchmark dataset...
Running on validation set with 24 documents


Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 156.11ba/s]

Saved seed data to: seed_data_val.jsonl


### Run SDG
- This will create knowledge flow from provided yaml file
- We will run this on small dataset for demo purposes
- For large scale generation, please use the python command provided in the next cell
- You can analyze the generated data to ensure the quality is similar to proivded QnA pairs

In [7]:
# Setup model configuration in flow object
def set_model_config(flow_object):
    model_provider = os.getenv('MODEL_PROVIDER', 'hosted_vllm')
    print(f"Using model provider: {model_provider}")
    # Set model provider
    if model_provider == 'hosted_vllm':    
        vllm_model = os.getenv('VLLM_MODEL', 'hosted_vllm/meta-llama/Llama-3.3-70B-Instruct')
        vllm_api_base = os.getenv('VLLM_API_BASE', 'http://localhost:8000/v1')
        vllm_api_key = os.getenv('VLLM_API_KEY', 'EMPTY')
        enable_reasoning = os.getenv('ENABLE_REASONING', False).lower() in ('1', 'true', 'yes')
        print(f"Using reasoning: {enable_reasoning}")
        flow_object.set_model_config(
            model=vllm_model,
            api_base=vllm_api_base,
            api_key=vllm_api_key,
            enable_reasoning=enable_reasoning,
        )
    elif model_provider == 'openai':
        openai_api_key = os.getenv('OPENAI_API_KEY')
        openai_model = os.getenv('OPENAI_MODEL', 'openai/gpt-4')
        flow_object.set_model_config(
            model=openai_model,
            api_key=openai_api_key,
        )
    elif model_provider == 'ollama':
        ollama_model = os.getenv('OLLAMA_MODEL', 'ollama/gemma2')
        ollama_api_base = os.getenv('OLLAMA_API_BASE', 'http://localhost:11434')
        flow_object.set_model_config(
            model=ollama_model,
            api_base=ollama_api_base,
        )
    elif model_provider == 'maas':
        maas_model = os.getenv('MAAS_MODEL')
        maas_api_base = os.getenv('MAAS_API_BASE')
        maas_api_key = os.getenv('MAAS_API_KEY')
        flow_object.set_model_config(
            model=maas_model,
            api_base=maas_api_base,
            api_key=maas_api_key,
        )
    return flow_object 

#### Discover the available generation flows

In [8]:
# Auto-discover all available flows (no setup needed!)
FlowRegistry.discover_flows()

# List available flows
flows = FlowRegistry.list_flows()
print(f"Available flows: {flows}")

# You can also search the flows by tag
qa_flows = FlowRegistry.search_flows(tag="question-generation")
print(f"QA flows: {qa_flows}")

[16:50:42] INFO     Discovered 5 flows                                                              ]8;id=85366;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/registry.py\registry.py]8;;\:]8;id=233172;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/registry.py#113\113]8;;\

┏━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ ID               ┃ Name                  ┃ Author               ┃ Tags                  ┃ Description           ┃
┡━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ epic-jade-656    │ Extractive Summary    │ SDG Hub Contributors │ knowledge-tuning,     │ Generate extractive   │
│                  │ Knowledge Tuning      │                      │ document-internaliza… │ summary from the      │
│                  │ Dataset Generation    │                      │ question-generation,  │ input document. Each  │
│                  │ Flow                  │                      │ knowledge-extractive… │ document is first     │
│                  │                       │                      │ qa-pairs,             │ converted into list   │
│                  │                       │                      │ extractive-summaries  │ of knowledge segments │
│                  │                       │                      │                       │ for creating          │
│                  │                       │                      │                       │ extractive summary    │
│                  │                       │                      │                       │ and then annotated    │
│                  │                       │                      │                       │ with context,         │
│                  │                       │                      │                       │ relationship and      │
│                  │                       │                      │                       │ relevance. This is    │
│                  │                       │                      │                       │ then converted into   │
│                  │                       │                      │                       │ Question-Answer       │
│                  │                       │                      │                       │ pairs.                │
│ green-clay-812   │ Structured Text       │ SDG Hub Contributors │ text-analysis,        │ Multi-step pipeline   │
│                  │ Insights Extraction   │                      │ summarization, nlp,   │ for extracting        │
│                  │ Flow                  │                      │ structured-output,    │ structured insights   │
│                  │                       │                      │ insights,             │ from text including   │
│                  │                       │                      │ sentiment-analysis,   │ summary, keywords,    │
│                  │                       │                      │ entity-extraction,    │ entities, and         │
│                  │                       │                      │ keyword-extraction    │ sentiment analysis    │
│                  │                       │                      │                       │ combined into a JSON  │
│                  │                       │                      │                       │ output                │
│ heavy-heart-77   │ Key Facts Knowledge   │ SDG Hub Contributors │ knowledge-tuning,     │ Generating list of    │
│                  │ Tuning Dataset        │                      │ document-internaliza… │ atomic facts from a   │
│                  │ Generation Flow       │                      │ question-generation,  │ document and          │
│                  │                       │                      │ qa-pairs, key-facts   │ converting each       │
│                  │                       │                      │                       │ atomic fact into a QA │
│                  │                       │                      │                       │ pair. This flow will  │
│                  │                       │                      │                       │ generate 5 QA pairs   │
│                  │                       │            

Available flows: [{'id': 'green-clay-812', 'name': 'Structured Text Insights Extraction Flow'}, {'id': 'small-rock-799', 'name': 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}]
QA flows: [{'id': 'small-rock-799', 'name': 'Advanced Document Grounded Question-Answer Generation Flow for Knowledge Tuning'}, {'id': 'mild-thunder-748', 'name': 'Detailed Summary Knowledge Tuning Dataset Generation Flow'}, {'id': 'heavy-heart-77', 'name': 'Key Facts Knowledge Tuning Dataset Generation Flow'}, {'id': 'epic-jade-656', 'name': 'Extractive Summary Knowledge Tuning Dataset Generation Flow'}]


In [9]:
# We will use below mapping of flow names to their respective summarization flows
flow_name_map = {
        'Detailed Summary Knowledge Tuning Dataset Generation Flow': 'gen_detailed_summary',
        'Key Facts Knowledge Tuning Dataset Generation Flow': 'gen_atomic_facts',
        'Extractive Summary Knowledge Tuning Dataset Generation Flow': 'gen_extractive_summary',
    }

In [10]:
# Get runtime parameters
enable_reasoning = os.getenv('ENABLE_REASONING', False).lower() in ('1', 'true', 'yes')
number_of_summaries = int(os.getenv('NUMBER_OF_SUMMARIES', '50'))

In [11]:
# Generate data for extractive summary
flow_name = "Extractive Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)
number_of_summaries = int(os.getenv('NUMBER_OF_SUMMARIES', '50'))
# Generate data for extractive summary
runtime_params = {
    flow_name_map[flow_name]: {
        'n': 2
    },
}
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params.update({'question_generation': {'max_tokens': 1024}, 'gen_extractive_summary': {'max_tokens': 6000}})
    

extractive_summary_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=100)
save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
extractive_summary_generated_data.to_json(os.path.join(save_data_path, 'extractive_summary', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Extractive summary: {len(extractive_summary_generated_data)} records")

print(f"✓ Columns: {list(extractive_summary_generated_data.column_names)}")

[16:50:53] INFO     Loading flow from:                                                                  ]8;id=324325;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=612149;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#172\172]8;;\
                    /Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/flows/qa_generation/document_gro            
                    unded_qa/enhanced_multi_summary_qa/extractive_summary/flow.yaml                                

Using model provider: hosted_vllm
Using reasoning: True


           INFO     Auto-detected 4 LLM blocks for configuration: ['answer_generation',                 ]8;id=51711;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=440643;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#864\864]8;;\
                    'eval_faithfulness', 'gen_extractive_summary', 'question_generation']                          

           WARNING  Block 'gen_extractive_summary' (LLMChatBlock) does not have attribute               ]8;id=384296;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=417726;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#881\881]8;;\
                    'enable_reasoning' - skipping                                                                  

[16:50:53] INFO     Loaded LLM client for model 'hosted_vllm/qwen3-32b'                        ]8;id=775298;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=851859;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#61\61]8;;\

...

[16:55:06] INFO     Block 'eval_faithfulness' completed successfully: 266 samples, 22 columns           ]8;id=595450;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=324940;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭──────────────────── Extractive Summary Knowledge Tuning Dataset Generation Flow - Complete ─────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ extractive_summary_… │ PromptBuilderB… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ gen_extractive_summ… │ LLMChatBlock    │     73.60s │    1 → 1     │       +1        │     ✓      │           │
│ │ parse_extractive_su… │ TextParserBlock │      0.08s │    1 → 50    │       +1        │     ✓      │           │
│ │ rename_to_document_… │ RenameColumnsB… │      0.00s │   50 → 50    │      +1/-1      │     ✓      │           │
│ │ question_generation… │ PromptBuilderB… │      0.02s │   50 → 50    │       +1        │     ✓      │           │
│ │ question_generation  │ LLMChatBlock    │     27.17s │   50 → 50    │       +1        │     ✓      │           │
│ │ parse_question_list  │ TextParserBlock │      0.44s │   50 → 266   │       +1        │     ✓      │           │
│ │ answer_generation_p… │ PromptBuilderB… │      0.06s │  266 → 266   │       +1        │     ✓      │           │
│ │ answer_generation    │ LLMChatBlock    │     89.77s │  266 → 266   │       +1        │     ✓      │           │
│ │ parse_response_dict  │ TextParserBlock │      0.70s │  266 → 266   │       +2        │     ✓      │           │
│ │ eval_faithfulness    │ EvaluateFaithf… │     60.62s │  266 → 266   │       +4        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 12 blocks       │    252.49s │  266 final   │    22 final     │   12/12    │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Extractive Summary Knowledge Tuning Dataset Generation Flow' completed        ]8;id=940336;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=992701;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\
                    successfully: 266 final samples, 22 final columns                                              

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  1.53ba/s]

✓ Extractive summary: 266 records
✓ Columns: ['document_outline', 'raw_document', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'base_document', 'extractive_summary_prompt', 'raw_summary', 'document', 'question_generation_prompt', 'question_list', 'question', 'answer_generation_prompt', 'response_dict', 'response', 'parse_response_dict_reasoning_content', 'eval_faithfulness_prompt', 'raw_eval_faithfulness', 'faithfulness_explanation', 'faithfulness_judgment']


In [12]:
# Generate similar data for Detailed Summary
flow_name = "Detailed Summary Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)

runtime_params.update({ flow_name_map[flow_name]: {
        'n': number_of_summaries
    }})
if enable_reasoning:
    # Increase max tokens to accommodate reasoning content
    runtime_params.update({'question_generation': {'max_tokens': 1024}, 'gen_detailed_summary': {'max_tokens': 6000}})
# Generate data for detailed summary
detailed_summary_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=100)
save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
detailed_summary_generated_data.to_json(os.path.join(save_data_path, 'detailed_summary', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Detailed summary: {len(detailed_summary_generated_data)} records")

print(f"✓ Columns: {list(detailed_summary_generated_data.column_names)}")

[16:55:17] INFO     Loading flow from:                                                                  ]8;id=746143;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=610872;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#172\172]8;;\
                    /Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/flows/qa_generation/document_gro            
                    unded_qa/enhanced_multi_summary_qa/detailed_summary/flow.yaml                                  

[16:55:17] INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=228436;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=864087;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

           INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=337413;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=813497;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

           INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=760871;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=604438;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

           INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=270821;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=596197;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

...

[16:58:50] INFO     Block 'eval_faithfulness' completed successfully: 281 samples, 22 columns           ]8;id=198348;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=679402;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭───────────────────── Detailed Summary Knowledge Tuning Dataset Generation Flow - Complete ──────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ duplicate_document_… │ DuplicateColum… │      0.04s │    1 → 1     │       +1        │     ✓      │           │
│ │ detailed_summary_pr… │ PromptBuilderB… │      0.05s │    1 → 1     │       +1        │     ✓      │           │
│ │ gen_detailed_summary │ LLMChatBlock    │     36.00s │    1 → 1     │       +1        │     ✓      │           │
│ │ parse_detailed_summ… │ TextParserBlock │      0.08s │    1 → 50    │       +1        │     ✓      │           │
│ │ rename_to_document_… │ RenameColumnsB… │      0.00s │   50 → 50    │      +1/-1      │     ✓      │           │
│ │ question_generation… │ PromptBuilderB… │      0.02s │   50 → 50    │       +1        │     ✓      │           │
│ │ question_generation  │ LLMChatBlock    │     25.08s │   50 → 50    │       +1        │     ✓      │           │
│ │ parse_question_list  │ TextParserBlock │      0.40s │   50 → 281   │       +1        │     ✓      │           │
│ │ answer_generation_p… │ PromptBuilderB… │      0.06s │  281 → 281   │       +1        │     ✓      │           │
│ │ answer_generation    │ LLMChatBlock    │     90.58s │  281 → 281   │       +1        │     ✓      │           │
│ │ parse_response_dict  │ TextParserBlock │      0.56s │  281 → 281   │       +2        │     ✓      │           │
│ │ eval_faithfulness    │ EvaluateFaithf… │     60.11s │  281 → 281   │       +4        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 12 blocks       │    212.99s │  281 final   │    22 final     │   12/12    │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Detailed Summary Knowledge Tuning Dataset Generation Flow' completed          ]8;id=319985;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=992302;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\
                    successfully: 281 final samples, 22 final columns                                              

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00,  1.79ba/s]

✓ Detailed summary: 281 records
✓ Columns: ['document_outline', 'raw_document', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'base_document', 'summary_prompt', 'raw_summary', 'document', 'question_generation_prompt', 'question_list', 'question', 'answer_generation_prompt', 'response_dict', 'response', 'parse_response_dict_reasoning_content', 'eval_faithfulness_prompt', 'raw_eval_faithfulness', 'faithfulness_explanation', 'faithfulness_judgment']


In [13]:
# Generate similar data for key facts 
flow_name = "Key Facts Knowledge Tuning Dataset Generation Flow"
flow_path = FlowRegistry.get_flow_path(flow_name)
flow = Flow.from_yaml(flow_path)

# Set model configuration
flow = set_model_config(flow)

runtime_params.update({ flow_name_map[flow_name]: {
        'n': 2
    }})


# Generate data for key facts summary
key_facts_generated_data = flow.generate(quality_corpus, runtime_params=runtime_params, max_concurrency=100)
save_data_path = os.getenv('OUTPUT_DATA_FOLDER', '')
key_facts_generated_data.to_json(os.path.join(save_data_path, 'key_facts_to_qa', 'gen.jsonl'), orient='records', lines=True)

print(f"✓ Key facts: {len(key_facts_generated_data)} records")

print(f"✓ Columns: {list(key_facts_generated_data.column_names)}")

[16:59:05] INFO     Loading flow from:                                                                  ]8;id=397782;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=112276;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#172\172]8;;\
                    /Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/flows/qa_generation/document_gro            
                    unded_qa/enhanced_multi_summary_qa/key_facts/flow.yaml                                         

[16:59:05] INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=558404;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=300271;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

           INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=933049;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=534890;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

           INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=177933;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=409591;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

           INFO     Unloaded LLM client for model 'hosted_vllm/qwen3-32b'                      ]8;id=871415;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py\client_manager.py]8;;\:]8;id=434490;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/blocks/llm/client_manager.py#75\75]8;;\

...

           INFO     Block 'parse_key_fact_qa' completed successfully: 190 samples, 15 columns           ]8;id=816761;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=527552;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#732\732]8;;\

╭───────────────────────── Key Facts Knowledge Tuning Dataset Generation Flow - Complete ─────────────────────────╮
│                                        Flow Execution Summary                                                   │
│ ┏━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┓           │
│ ┃ Block Name           ┃ Type            ┃   Duration ┃     Rows     ┃     Columns     ┃   Status   ┃           │
│ ┡━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━┩           │
│ │ atomic_facts_prompt  │ PromptBuilderB… │      0.01s │    1 → 1     │       +1        │     ✓      │           │
│ │ gen_atomic_facts     │ LLMChatBlock    │     39.36s │    1 → 1     │       +1        │     ✓      │           │
│ │ parse_atomic_facts   │ TextParserBlock │      0.01s │    1 → 2     │       +1        │     ✓      │           │
│ │ parse_atomic_facts_… │ TextParserBlock │      0.06s │    2 → 38    │       +1        │     ✓      │           │
│ │ rename_to_document_… │ RenameColumnsB… │      0.00s │   38 → 38    │      +1/-1      │     ✓      │           │
│ │ key_fact_qa          │ PromptBuilderB… │      0.03s │   38 → 38    │       +1        │     ✓      │           │
│ │ generate_key_fact_qa │ LLMChatBlock    │     22.15s │   38 → 38    │       +1        │     ✓      │           │
│ │ parse_key_fact_qa    │ TextParserBlock │      0.04s │   38 → 190   │       +2        │     ✓      │           │
│ ├──────────────────────┼─────────────────┼────────────┼──────────────┼─────────────────┼────────────┤           │
│ │ TOTAL                │ 8 blocks        │     61.65s │  190 final   │    15 final     │    8/8     │           │
│ └──────────────────────┴─────────────────┴────────────┴──────────────┴─────────────────┴────────────┘           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

           INFO     Flow 'Key Facts Knowledge Tuning Dataset Generation Flow' completed successfully:   ]8;id=531033;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py\base.py]8;;\:]8;id=516155;file:///Users/mathale/redhat-projects/sdg_hub/src/sdg_hub/core/flow/base.py#620\620]8;;\
                    190 final samples, 15 final columns                                                            

Creating json from Arrow format: 100%|██████████| 1/1 [00:00<00:00, 14.33ba/s]

✓ Key facts: 190 records
✓ Columns: ['document_outline', 'raw_document', 'icl_document', 'icl_query_1', 'icl_query_2', 'icl_query_3', 'domain', 'atomic_facts_prompt', 'raw_summary', 'document', 'key_fact', 'key_fact_qa', 'raw_key_fact_qa', 'question', 'response']


🎉 You now have all three types of document augmentations (detailed summaries, extractive summaries, and key facts) along with their corresponding QA pairs.

✅ Next steps:
   - Combine and curate these datasets to prepare your final training data.
   - For detailed guidance on post-processing, mixing, and formatting the data for model training (including conversion to messages format), please refer to [knowledge_mixing.ipynb](knowledge_mixing.ipynb).